# 影评分类：二分类问题示例

对应 Keras 版本：`Keras应用/第四章-神经网络入门：分类与回归/IMDB影评二分类.ipynb`

使用 PyTorch 完成 IMDB 电影评论情感分析：
- 输入：电影评论（编码为整数序列）
- 输出：正面评价 (1) 还是 负面评价 (0)
- 类别数：2 类

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")

## 1. IMDB 数据集

使用 torchvision 或手动加载 IMDB 数据集。这里使用与 Keras 版本相同的数据源。

In [ ]:
# 加载 IMDB 数据（与 Keras 相同的数据源）
from tensorflow.keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

print(f"训练集样本数: {len(train_data)}")
print(f"测试集样本数: {len(test_data)}")
print(f"\n第一条训练样本（整数序列）: {train_data[0][:20]}...")
print(f"第一条训练样本标签: {train_labels[0]} (0=负面, 1=正面)")

In [ ]:
# 解码查看原始文本
word_index = imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_review = " ".join([reverse_word_index.get(i - 3, "?") for i in train_data[0]])
print(decoded_review[:300] + "...")

## 2. 数据预处理：Multi-hot 编码

每条评论长度不同，神经网络需要固定形状输入。
使用 multi-hot 编码：创建 10000 维向量，出现过的词对应位置为 1。

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    """将整数序列列表转为 multi-hot 编码的 numpy 数组"""
    results = np.zeros((len(sequences), dimension), dtype=np.float32)
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

# 向量化
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = np.asarray(train_labels, dtype=np.float32)
y_test = np.asarray(test_labels, dtype=np.float32)

print(f"x_train 形状: {x_train.shape}")
print(f"x_test 形状: {x_test.shape}")
print(f"\n第一条评论向量化后（前20维）: {x_train[0][:20]}")

## 3. 构建模型

与 Keras 版本完全相同的结构：
```
Input(10000) → Dense(16, relu) → Dense(16, relu) → Dense(1, sigmoid)
```

In [ ]:
class IMDBClassifier(nn.Module):
    """
    IMDB 二分类模型。
    结构对应 Keras 版本：Dense(16) → Dense(16) → Dense(1, sigmoid)
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10000, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))  # 输出 0~1 概率
        return x.squeeze(-1)  # 去掉最后一维，变为 (batch,)

model = IMDBClassifier()
print(model)
print(f"\n总参数: {sum(p.numel() for p in model.parameters())}")

## 4. 划分验证集

In [ ]:
# 划分验证集（前 10000 条）
x_val = torch.from_numpy(x_train[:10000])
y_val = torch.from_numpy(y_train[:10000])
x_train_partial = torch.from_numpy(x_train[10000:])
y_train_partial = torch.from_numpy(y_train[10000:])

print(f"训练集: {x_train_partial.shape[0]} 样本")
print(f"验证集: {x_val.shape[0]} 样本")

## 5. 训练循环

In [ ]:
# 训练函数
def train_epoch(model, x, y, optimizer, loss_fn, batch_size=512):
    model.train()
    total_loss = 0
    correct = 0
    n_samples = len(x)
    
    indices = torch.randperm(n_samples)
    x, y = x[indices], y[indices]
    
    for i in range(0, n_samples, batch_size):
        batch_x = x[i:i+batch_size]
        batch_y = y[i:i+batch_size]
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = loss_fn(output, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        correct += ((output >= 0.5) == batch_y).sum().item()
    
    return total_loss / n_samples, correct / n_samples


@torch.no_grad()
def evaluate(model, x, y, loss_fn):
    model.eval()
    output = model(x)
    loss = loss_fn(output, y)
    correct = ((output >= 0.5) == y).sum().item()
    return loss.item(), correct / len(y)


# 训练
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()  # 二分类交叉熵

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

epochs = 20
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, x_train_partial, y_train_partial,
                                        optimizer, loss_fn, batch_size=512)
    val_loss, val_acc = evaluate(model, x_val, y_val, loss_fn)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch+1:2d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"val_acc={val_acc:.4f}")

## 6. 绘制训练曲线

In [ ]:
epochs_range = range(1, epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], "bo", label="Training loss")
axes[0].plot(epochs_range, history["val_loss"], "b", label="Validation loss")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and validation loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, history["train_acc"], "bo", label="Training acc")
axes[1].plot(epochs_range, history["val_acc"], "b", label="Validation acc")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training and validation accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("过拟合观察：验证损失大约在第4轮后开始上升，说明训练轮数不宜过多。")

## 7. 重新训练（选择合适轮数）

In [ ]:
# 重新训练，只训练 4 轮
model = IMDBClassifier()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

# 使用全部训练集
x_train_full = torch.from_numpy(x_train)
y_train_full = torch.from_numpy(y_train)

for epoch in range(4):
    train_loss, train_acc = train_epoch(model, x_train_full, y_train_full,
                                        optimizer, loss_fn, batch_size=512)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")

## 8. 测试集评估

In [ ]:
# 在测试集上评估
x_test_tensor = torch.from_numpy(x_test)
y_test_tensor = torch.from_numpy(y_test)

test_loss, test_acc = evaluate(model, x_test_tensor, y_test_tensor, loss_fn)
print(f"测试损失: {test_loss:.4f}")
print(f"测试准确率: {test_acc:.4f}")

## 9. 预测

In [ ]:
with torch.no_grad():
    predictions = model(x_test_tensor)
    predicted_labels = (predictions >= 0.5).long()

print("前20条预测概率：")
print(predictions[:20].numpy())

print("\n前20条预测类别：")
print(predicted_labels[:20].numpy())

### 与 Keras 版本对比

| 步骤 | Keras | PyTorch |
|------|-------|---------|
| 模型定义 | `Sequential([Dense(16), ...])` | `nn.Module` + `forward()` |
| 编译 | `compile(optimizer, loss, metrics)` | 创建 `optimizer` + `BCELoss()` |
| 训练 | `fit(x, y, epochs, batch_size)` | 手写 epoch 循环 |
| 验证 | `validation_data=(x_val, y_val)` | 手写 `evaluate()` 函数 |
| 预测 | `model.predict(x)` | `model(x)` |
| 损失函数 | `binary_crossentropy` | `nn.BCELoss()` |